🌐 [English](dx_app.ipynb) | **한국어** | [日本語](dx_app.ja.ipynb) | [中文](dx_app.zh.ipynb)

# DEEPX Tutorial 02 - DX-APP 사용법

이 두 번째 튜토리얼에서는 DX-APP을 소개하고, 컴파일된 DXNN 모델을 AI 애플리케이션에서 실행하는 방법을 배웁니다.

## 학습 목표

이 튜토리얼을 마치면 다음을 할 수 있습니다.

- DX-APP이 **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/)의 모든 모델**에 대해 바로 실행할 수 있는 C++ 및 Python 예제 코드를 제공한다는 것을 알고, 그 코드로 원하는 모델을 실행할 수 있습니다.
- DX-APP이 무엇이고 C++ 및 Python 예제가 어떻게 구성되어 있는지 설명할 수 있습니다.
- 튜토리얼에 필요한 모델과 영상만 다운로드할 수 있습니다.
- 이미지, 영상, 카메라를 입력으로 분류, 검출, 포즈, 세그멘테이션 예제를 실행할 수 있습니다.
- 동기, 비동기, PPU 실행 방식 중에서 선택할 수 있습니다.
- 대화형 데모 런처를 시작할 수 있습니다.
- 독립 실행형 C++ 또는 Python 예제를 내보내 DX-APP 트리 밖에서 실행할 수 있습니다.

## DX-APP이란?

**DX-APP**은 DEEPX NPU용 참조 애플리케이션입니다. 가장 큰 특징은 커버리지입니다. **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/)에 공개된 모든 모델에는 바로 실행할 수 있는 예제**가 C++과 Python 양쪽으로 준비되어 있으며, 작업(object detection, classification, face detection, pose, segmentation, depth 등)별로 정리되어 있습니다. Model Zoo 페이지에서 모델을 고르면 DX-APP에 이미 그 모델을 실행하는 코드가 들어 있습니다. DX-APP은 개발자가 런타임 환경을 빠르게 구성하도록 돕고,
자신만의 AI 애플리케이션을 만들고 커스터마이즈하기 위한 템플릿 역할을 합니다.

자세한 내용은 [dx_app git 저장소](https://github.com/DEEPX-AI/dx_app/blob/main/README.md)의 개요를 참고하거나, [여기](https://developer.deepx.ai/download/?id=580)에서 DX-APP User Guide를 다운로드하세요!

> **참고:** User Guide를 다운로드하려면 먼저 https://developer.deepx.ai/ 에 로그인해야 합니다.

DX-APP의 파일 구조를 살펴보겠습니다.

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** 데모는 최소한의 설정으로 DEEPX NPU에서 사전 컴파일된 모델을 시연하도록 최적화되어 있습니다.
각 데모는 일반적인 AI 작업을 나타내며 이미지, 영상, 실시간 카메라
입력으로 실행할 수 있습니다.

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP 저장소 구성:**
```shell
dx_app/
├── src/
│   ├── cpp_example/            # C++ end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Model-specific factories, configs, and entry points
│   │   └── common/             # Shared C++ runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   42 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   16 task-specific sync/async runner pairs
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   13 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader
│   │       ├── trackers/       #   Shared IoU tracker
│   │       ├── third_party/    #   Vendored header dependencies
│   │       └── utility/        #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/         # Python end-to-end examples (352 models across 22 tasks)
│   │   ├── <task>/<model>/     #   Four execution/postprocessing variants per model
│   │   └── common/             # Shared Python runtime layer
│   │       ├── base/           #   Abstract interfaces (IFactory, IProcessor, ...)
│   │       ├── processors/     #   41 postprocessors, 4 preprocessors, and helpers
│   │       ├── runner/         #   SyncRunner, AsyncRunner, arguments, and run utilities
│   │       ├── inputs/         #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/    #   14 task-specific visualizers
│   │       ├── config/         #   ModelConfig loader and schema
│   │       ├── trackers/       #   Shared IoU tracker
│   │       └── utility/        #   Labels, preprocessing, profiling, and geometry helpers
│   ├── postprocess/            # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           └── dx_postprocess/ # pybind11 bindings wrapping src/postprocess/
├── config/
│   ├── model_registry.json     # Model registry — single source of truth
│   ├── test_models.conf        # Test model configuration
│   └── README.md               # Config directory documentation
├── scripts/                    # Developer tools, validation, and helper scripts
├── tests/                      # C++, Python, script, helper, and Windows tests
├── assets/                     # Links to downloaded models and videos
├── sample/                     # Bundled images and task-specific sample data
├── bin/                        # C++ executables generated by the build
├── cmake/                      # x86_64 and aarch64 toolchain files
├── extern/                     # External header dependencies
├── build.sh / build.bat        # Linux and Windows build scripts
├── setup.sh / setup.bat        # Model, video, and dependency setup
├── setup_sample_models.sh      # Download the sample model set only
├── setup_sample_videos.sh      # Download the sample video set only
├── run_demo.sh / run_demo.bat  # Interactive demo launcher
├── run_tc.sh                   # Unified test runner for example tests
├── install.sh                  # Dependency and OpenCV installer
└── docs/                       # Detailed documentation
```
---
**DX-APP은 C++과 Python 예제를 모두 제공합니다:**

`cpp_example`과 `python_example`은 모두 작업(task)을 기준으로 한 동일한 구조를 따릅니다.

현재 두 언어 모두 동일한 22개의 작업 디렉터리를 제공합니다.

- 3d_object_detection/
- attribute_recognition/
- classification/
- depth_estimation/
- embedding/
- face_alignment/
- face_detection/
- hand_detection/
- hand_landmark/
- image_denoising/
- image_enhancement/
- instance_segmentation/
- keypoint_detection/
- obb_detection/
- object_detection/
- object_pose_estimation/
- panoptic_driving_perception/
- pose_estimation/
- ppu/
- reid/
- semantic_segmentation/
- super_resolution/

## 사전 요구 사항

- 튜토리얼 01 완료: DX-Runtime을 `--all`로 설치(DX-RT, DX-APP 및 그 `venv-dx-runtime` Python 환경)하고 NPU가 `/dev/dxrt0`으로 인식되는 상태.
- DX-APP 샘플 모델과 영상. 튜토리얼 01의 3.3절에서 다운로드하며, 아래 1절은 아직 없는 것만 받아옵니다(이 튜토리얼용 모델 약 300 MB, 새 호스트라면 공유 샘플 영상 아카이브 1.2 GB 추가).
- 그래픽 데스크톱 세션은 선택 사항입니다. 2절의 데모 셀은 도움말을 출력하거나 헤드리스로 실행되고, 3절은 독립 실행형 예제를 헤드리스로 실행한 뒤 저장된 이미지를 노트북에 표시합니다.
- `sudo` 불필요. 소요 시간은 약 30분이며, 3.3절의 C++ 빌드에 1~2분이 걸립니다.

다음 셀은 SDK를 찾고, 이 요구 사항을 확인한 뒤 상태 표를 출력합니다. `MISSING`으로 표시된 항목에는 그것을 제공하는 단계가 함께 표시됩니다.

## 1. 모델과 영상 준비

1. `dx_app` 디렉터리로 이동합니다.

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. `setup.sh`로 필요한 모델과 샘플 영상을 다운로드합니다. 이 스크립트는 대화형(어떤 모델 카테고리를 다운로드할지 묻습니다)이므로 셀이 아니라 터미널에서 실행합니다. **File > New > Terminal**을 열고 다음을 실행하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 다음 셀은 실제 설치 경로를 넣어 같은 두 명령을 출력합니다.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. 여유 저장 공간이 29 GB 이상이라면 **Enter**를 눌러 모든 Model Zoo 모델을 다운로드합니다.

5. 그렇지 않다면 아래 코드 셀로 이 튜토리얼에 필요한 모델만 다운로드하세요. 튜토리얼 01의 3.3절에서 이 목록을 포함하는 24개 모델을 이미 다운로드했으므로, 그 절을 완료했다면 `--no-force` 덕분에 셀이 모든 파일을 건너뛰고 몇 초 만에 끝납니다. 뒤의 3절에서는 같은 방식으로 거기서 사용하는 YOLO26 모델 두 개를 추가합니다.

In [ ]:
# The models used in this tutorial. Tutorial 01 section 3.3 already downloads them;
# this cell only downloads what is still missing (--no-force skips existing files).
SELECTED_MODELS = shlex.join([
    "ResNet50",
    "YoloV7",
    "YoloV7_PPU",
    "yolov8s_pose",
    "yolov8n_seg",
])

MODEL_MANIFEST_PATH = DX_APP_DIR / "scripts" / "modelzoo_manifest.json"

# Check the downloadable model list
#!cat "{MODEL_MANIFEST_PATH}" | grep name

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force


6. 모델과 영상이 모두 정상적으로 다운로드되었는지 확인합니다.

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. 데모 실행

**DX-APP**은 분류, 검출, 세그멘테이션, 포즈 추정 등 컴파일된 모델을 DEEPX NPU에서 실행하는 방법을 보여 주는 바로 사용할 수 있는 데모 애플리케이션 모음입니다.

사용자는 다양한 DX-APP 데모를 살펴보며 애플리케이션을 배우고 만들 수 있습니다.

### 2.1 분류
- **bin/resnet50_async**는 사전 학습된 이미지 분류 모델을 실행해 가장 가능성이 높은 클래스를 출력하는 DX-APP 실행 파일입니다.

In [ ]:
%cd {DX_APP_DIR} 

#!ls bin
!./bin/resnet50_async -h
#!./bin/resnet50_async -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg

### 2.2 객체 검출
- **bin/yolov\<version>_async**는 YOLO 기반 객체 검출 모델을 실행해 이미지, 영상 또는 카메라에서 객체를 검출하는 DX_APP 데모 바이너리입니다.

In [ ]:
%cd {DX_APP_DIR} 

# Find other yolo based reference excutable files
!ls ./bin/yolo*

#### 실행 방식

DX-APP은 다양한 애플리케이션 요구 사항에 맞는 여러 실행 방식을 제공합니다.

| 방식 | 주요 특징 | 권장 용도 |
|---|---|---|
| 동기(Synchronous) | 각 프레임을 순차적으로 처리 | 단순한 흐름과 지연 시간 확인 |
| 비동기(Asynchronous) | 파이프라인 단계를 겹쳐서 실행 | 더 높은 처리량 |
| PPU | 지원되는 후처리 작업을 오프로드 | 호스트 CPU 부하 감소 |

- **동기 애플리케이션**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_sync -h

# Image input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **비동기 애플리케이션**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

In [ ]:
%cd {DX_APP_DIR} 

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov7_async -h

# Image input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg

# Video input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4

# Camera input
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (This IP address only works in Korea)
#!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU와 비PPU 비교**

아래 두 셀은 같은 7초짜리 영상을 비PPU YOLOv7 모델과 PPU YOLOv7 모델로 각각 실행합니다. 기본값인 `SHOW_WINDOW = False`에서는 `--no-display`로 헤드리스 실행되어 FPS 요약만 출력하므로 노트북이 멈추지 않습니다. 데스크톱 세션에서 `SHOW_WINDOW = True`로 설정하면 DX-APP 창에서 검출 결과와 FPS 카운터를 볼 수 있습니다. 두 FPS 줄을 비교해 보세요. PPU 모델은 후보 필터링과 클래스 선택을 NPU로 옮깁니다.

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}


### 2.3 포즈 추정
- **bin/yolov8s_pose_sync**는 포즈 추정 모델을 실행해 이미지, 영상 또는 카메라에서 사람을 검출하고 키포인트를 추정하는 DX_APP 데모 바이너리입니다.

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8s_pose_sync -h

# Image input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov

# Camera input
#!./bin/yolov8s_pose_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 세그멘테이션
- **bin/yolov8n_seg_async**는 인스턴스 세그멘테이션 모델을 실행해 이미지, 영상 또는 카메라에 대해 픽셀 단위 클래스 레이블을 생성하는 DX_APP 데모 바이너리입니다.

In [ ]:
%cd {DX_APP_DIR}

# Uncomment one command line at a time, comment out the others, and test each input.

# Help
!./bin/yolov8n_seg_async -h

# Image input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg

# Video input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4

# Camera input
#!./bin/yolov8n_seg_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 데모 스크립트

`run_demo.sh`는 C++ 데모와 Python 데모를 모두 실행할 수 있는 대화형 런처입니다.

대화형이므로 터미널(**File > New > Terminal**)에서 실행하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

다음 셀은 실제 경로를 넣어 같은 명령을 출력합니다.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


> ![](assets/sc-run_demo.png)

런처는 DX-APP 작업 카테고리 전반에 걸쳐 바로 실행할 수 있는 데모 23개를 제공합니다.

`0` Object Detection을 선택하면 다음 실행 옵션을 사용할 수 있습니다.
> ![](assets/sc-run_demo2.png)

옵션 [1, 2]는 C++ 기반 애플리케이션을 실행하고, 옵션 [3–6]은 Python 기반 애플리케이션을 실행합니다.

> **참고:** Python 기반 애플리케이션을 실행하려면 `dx-engine`이 설치된 Python 가상 환경을 활성화하세요.

예시:
> ![](assets/sc-run_demo3.png)

### 2.6 C++ 및 Python으로 작성된 샘플 애플리케이션 추출

`scripts/dx_tool.sh`는 예제를 관리하고 내보내는 대화형 터미널 도구입니다. `extract` 메뉴는 한 모델의 예제를 공유 `common/` 런타임 레이어와 함께 복사해, DX-APP 트리 **밖에서** 빌드하고 실행할 수 있는 자체 완결형 패키지를 만듭니다. 터미널(File > New > Terminal)에서 실행하세요.

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

다음 셀은 실제 경로를 넣어 같은 명령을 출력합니다. 메뉴가 나타나면 그 아래의 단계를 따르세요.

이 메뉴는 내부적으로 `scripts/extract_model_package.sh`를 호출합니다. 3절에서는 이 스크립트를 직접 사용하는데, 모델과 언어를 인자로 받기 때문에 노트북 셀 안에서 실행할 수 있기 때문입니다.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


`dx_tool.sh`를 실행하면 다음 메뉴가 표시됩니다. 
- `3`을 선택해 관심 있는 모델 목록을 확인합니다.
- `2`를 선택해 샘플 코드를 추출합니다.
- 프로그래밍 언어를 선택합니다. (C++, Python)
- 애플리케이션에 사용할 카테고리와 모델을 입력합니다. (예: `object_detection/yolo26s`)

> ![](assets/sc-extract.png)

몇 초 뒤 `output` 디렉터리에 샘플 코드가 생성됩니다.

> ![](assets/sc-extract2.png)

## 3. 추출한 독립 실행형 예제 실행

2.6절에서는 예제를 대화형으로 내보냈습니다. 이 절에서는 같은 작업을 노트북에서 `scripts/extract_model_package.sh`로 수행한 뒤, 내보낸 패키지를 **DX-APP 트리 밖에서** 빌드하고 실행합니다. 여러분의 프로젝트에서 하는 것과 똑같은 방식입니다. 세 예제가 두 언어와, 네 가지 Python 변형 중 두 가지를 다룹니다.

| 예제 | 모델(작업) | 언어 및 변형 | 보여 주는 내용 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26s` | Python, `*_sync.py` | Model Zoo 모델을 실행하는 가장 간단한 방법 |
| 3.3 | `object_detection/yolo26s` | C++, `*_sync` | 내보낸 C++ 패키지를 CMake로 빌드 |
| 3.4 | `pose_estimation/yolo26s_pose` | Python, `*_async_cpp_postprocess.py` | 다른 작업, 처리량이 가장 높은 변형 |

내보낸 모든 Python 예제는 DX-Runtime이 `dx-runtime/venv-dx-runtime`에 설치한 `dx_engine`과 `dx_postprocess`를 임포트합니다. 따라서 셀은 그 인터프리터를 명시적으로 호출합니다. 내보낸 예제에는 샘플 이미지가 없으므로, 셀은 `dx_app/sample/img`의 이미지를 전달해 헤드리스(`--no-display --save`)로 실행한 뒤 저장된 결과를 노트북에 표시합니다.

### 3.1 패키지 내보내기

먼저 스크립트 자체의 도움말부터 확인합니다. 다음 셀에서 사용하는 인자인 `<task>/<model>` 형식의 모델, `--lang cpp|py|both`, `--output-dir`이 나열됩니다.

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


`--output-dir` 없이 실행하면 스크립트가 `dx_app` 안에 패키지를 그대로 준비하고, 지정하면 패키지가 `<dir>/cpp/<task>/<model>`과 `<dir>/py/<task>/<model>`로 복사됩니다. 다음 셀은 세 패키지를 이 튜토리얼의 `workspace/` 디렉터리로 내보내며, 이 디렉터리는 git이 무시합니다.

In [ ]:
from IPython.display import Image, display

STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26s yolo26s-pose --no-force

# Export: object detection in both languages, pose estimation in Python.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26s --lang both --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26s_pose --lang py --output-dir "{STANDALONE_DIR}"

!tree -L 3 "{STANDALONE_DIR}"


### 3.2 예제 1: Python 독립 실행형, 객체 검출

`yolo26s_sync.py`는 가장 기본적인 동기 변형으로, 순수 Python 후처리를 사용해 한 번에 한 프레임씩 처리합니다. 가장 먼저 읽어 볼 변형입니다. 인자는 DX-APP의 모든 Python 예제에서 동일합니다. `--model`, 입력 옵션 하나(`--image`, `--video`, `--camera`, `--rtsp`), 그리고 헤드리스 실행을 위한 `--no-display --save`입니다.

```bash
cd <workspace>/standalone/py/object_detection/yolo26s
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26s"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_sync-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.3 예제 2: C++ 독립 실행형, 객체 검출

내보낸 C++ 패키지에는 `yolo26s_sync.cpp`, `yolo26s_async.cpp`, 공유 `common/` 레이어, 그리고 설치된 DX-RT에 링크하는 생성된 `CMakeLists.txt`가 들어 있습니다. 빌드는 표준 CMake 절차이며, DX-APP 트리를 참조하는 것은 아무것도 없습니다.

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26s
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26s_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

C++ 예제는 옵션을 `-m/--model_path`와 `-i/--image_path`로 표기한다는 점에 유의하세요. `./yolo26s_sync --help`로 목록을 볼 수 있습니다. 빌드에는 1~2분이 걸립니다.

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26s"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26s_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("Yolo26s_sync-*/output.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.4 예제 3: Python 독립 실행형, 포즈 추정, C++ 후처리를 사용하는 비동기

모든 Python 예제는 네 가지 변형으로 제공됩니다. 파일 이름을 보면 어떤 변형을 실행하는지 알 수 있습니다.

| 변형 | 후처리 | 스레딩 | 용도 |
|---|---|---|---|
| `*_sync.py` | 순수 Python | 동기 | 로직 학습과 디버깅 |
| `*_async.py` | 순수 Python | 비동기 | 첫 번째 성능 개선 단계 |
| `*_sync_cpp_postprocess.py` | C++ 바인딩(`dx_postprocess`) | 동기 | CPU 후처리 부하가 큰 경우 |
| `*_async_cpp_postprocess.py` | C++ 바인딩(`dx_postprocess`) | 비동기 | 최대 FPS. 배포용 변형 |

이 예제는 다른 작업과 가장 빠른 변형으로 전환합니다. 명령 형태는 바뀌지 않으며, 디렉터리, 스크립트 이름, 모델만 달라집니다.

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26s_pose
<dx-runtime>/venv-dx-runtime/bin/python yolo26s_pose_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26s_pose"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26s_pose_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

result = max(OUTPUT_DIR.glob("yolo26s_pose_async_cpp_postprocess-*/*_result.jpg"), key=lambda p: p.stat().st_mtime)
print("Saved result:", result)
display(Image(filename=str(result), width=640))


### 3.5 직접 해 보기: Model Zoo의 아무 모델이나

세 예제는 하나의 패턴을 따랐고, 이 패턴은 352개 모델 전부에 적용됩니다.

1. [Model Zoo](https://developer.deepx.ai/modelzoo/) 페이지에서 모델을 고르고, `./scripts/dx_tool.sh search <keyword>` 또는 `./scripts/dx_tool.sh list <category>`로 DX-APP에서의 이름을 찾습니다.
2. 다운로드: `./setup.sh --models <name> --no-force`.
3. 내보내기: `scripts/extract_model_package.sh <task>/<name> --lang py --output-dir <dir>`.
4. 실행: `<dir>/py/<task>/<name>/<name>_sync.py --model <file>.dxnn --image <image>`.

다음 셀은 `config/model_registry.json`에서 모델을 찾아 그 모델용 네 가지 명령을 출력합니다. `MODEL_NAME`을 시도해 보고 싶은 모델로 바꾼 뒤, 출력된 명령을 셀이나 터미널에서 실행하세요. 예를 들어 `scrfd500m`(face detection, `sample_face.jpg` 사용)이나 `resnet50`(classification, `sample/ILSVRC2012`의 이미지 사용)을 써 볼 수 있습니다.

In [ ]:
import json

MODEL_NAME = "scrfd500m"   # change me: any model_name from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if e["model_name"] == MODEL_NAME), None)
if entry is None:
    raise KeyError(f"{MODEL_NAME} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {MODEL_NAME}")

task, dxnn = entry["add_model_task"], entry["dxnn_file"]
print(f"{MODEL_NAME}: task={task}, model file={dxnn}\n")
print(f'cd "{DX_APP_DIR}" && bash setup.sh --models {MODEL_NAME} --no-force')
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{MODEL_NAME} --lang py --output-dir "{STANDALONE_DIR}"')
print(f'cd "{STANDALONE_DIR}/py/{task}/{MODEL_NAME}" && "{DX_RUNTIME_PYTHON}" {MODEL_NAME}_sync.py --model "{MODEL_DIR / dxnn}" --image <an image under {SAMPLE_IMG_DIR}> --no-display --save --save-dir "{OUTPUT_DIR}"')


## 4. 문제 해결

| 증상 | 표시되는 내용 | 원인 | 해결 |
|---|---|---|---|
| 실행 파일이 없음 | `./bin/yolov7_async: No such file or directory` | DX-APP이 빌드되지 않았거나 해당 타깃 없이 빌드됨 | 터미널에서 `dx_app`의 `./build.sh`(또는 `./build.sh --target yolov7_async`) 실행 |
| 모델을 불러올 수 없음 | `assets/models` 아래 경로와 함께 `Failed to load model` | 모델이 다운로드되지 않음 | 1절의 다운로드 셀을 해당 모델 이름으로 실행한 뒤 다시 실행 |
| 출력 창이 없음 | `cannot open display` 또는 창 없이 셀이 종료됨 | 그래픽 세션 없음(SSH 또는 헤드리스 호스트) | 데스크톱 세션에서 실행하거나, 이미지 입력(`-i`)을 사용하고 저장된 결과 확인 |
| 카메라를 열 수 없음 | `Could not open camera 0` | `/dev/video0`이 없거나 사용자가 `video` 그룹에 속해 있지 않음 | `v4l2-ctl --list-devices`; `sudo usermod -aG video $USER` 실행 후 다시 로그인 |
| `run_demo.sh`가 데모를 저절로 시작함 | 10초 뒤 옵션 0이 실행됨 | 런처에 10초 입력 제한 시간이 있음 | 메뉴가 나타나는 즉시 옵션 번호 입력 |
| 내보낸 Python 예제가 `dx_engine`을 임포트하지 못함 | `ModuleNotFoundError: No module named 'dx_engine'` | DX-Runtime 환경 대신 Jupyter Python으로 실행됨 | 3절의 셀처럼 `<dx-runtime>/venv-dx-runtime/bin/python` 사용 |
| 내보낸 C++ 패키지가 구성(configure)되지 않음 | `Could not find a package configuration file provided by "dxrt"` | DX-RT가 설치되지 않았거나 사용자 지정 prefix 아래에 있음 | DX-Runtime 설치(튜토리얼 01의 3절) 또는 `cmake` 줄에 `-DCMAKE_PREFIX_PATH=<prefix>` 추가 |
| 내보낸 예제가 입력을 찾지 못함 | `Input file not found: sample/img/...` | 패키지에 샘플 이미지가 없음 | `dx_app/sample/img`의 파일을 `--image`(Python) 또는 `-i`(C++)로 전달 |

## 5. 요약

### 5.1 완료한 DX-APP 워크플로

```text
Load SDK paths
       │
       ▼
Prepare selected models and videos
       │
       ▼
Run C++ or Python examples
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Use run_demo.sh or export a standalone example
```

### 5.2 학습 대시보드

| 영역 | 이제 할 수 있는 것 |
|---|---|
| SDK 경로 | `config.json`에서 `DX_APP_DIR`과 관련 경로 불러오기 |
| 모델 설정 | 실습에 필요한 모델과 영상만 다운로드 |
| 예제 선택 | 작업, 모델, 언어, 실행 방식별로 예제 찾기 |
| 실행 방식 | 목표에 맞게 동기, 비동기, PPU 실행 중 선택 |
| 데모 런처 | `run_demo.sh`로 설치된 예제 시작 |
| 독립 실행형 코드 | `scripts/extract_model_package.sh`로 아무 모델의 예제나 내보낸 뒤 DX-APP 밖에서 빌드하고 실행 |
| Model Zoo 커버리지 | `config/model_registry.json` 또는 `dx_tool.sh search`로 모든 Model Zoo 모델의 DX-APP 예제 찾기 |

### 5.3 완료 체크리스트

- [ ] DX-RT에서 DEEPX NPU가 인식되는지 확인
- [ ] 필요한 DX-APP 모델과 영상 에셋 준비
- [ ] 엔드투엔드 추론 예제를 하나 이상 실행
- [ ] 사용 가능한 실행 방식 비교
- [ ] 대상 작업의 참조 구현 찾기
- [ ] `run_demo.sh` 열기
- [ ] 독립 실행형 Python 및 C++ 예제를 DX-APP 트리 밖에서 내보내고 빌드하고 실행
- [ ] 직접 선택한 Model Zoo 모델 실행

> **다음:** 튜토리얼 03으로 이동해 학습된 모델을 조정하고, DXNN으로 컴파일하고, DX-APP 런타임 구성과 통합해 보세요.